<a href="https://colab.research.google.com/github/geeta-gwalior/Gemma-with-RAG-A-Coffee-Shop-Chatbot/blob/main/Gemma4_handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

# Set Kaggle username and key from Colab secrets
os.environ["KAGGLE_USERNAME"] = userdata.get("kaggle_user")
os.environ["KAGGLE_KEY"] = userdata.get("kaggle_key")

print("✅ Done!")

✅ Done!


In [ ]:
# Install necessary libraries
!pip install -q kagglehub
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate torch

# IMPORTANT: Restart the runtime after installation!
print("✅ Done! Please restart the Colab runtime now (Runtime → Restart runtime).")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Done! Ab Runtime → Restart Runtime karo


In [ ]:
# Check if a GPU is available and display its details
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.6 GB


In [ ]:
# Verify the installed Transformers library version
import transformers
print(transformers.__version__)
# The version should be 4.52 or higher

5.8.0.dev0


In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
# Load the model with 4-bit quantization to save memory
import kagglehub
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the path to the model from KaggleHub
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

# Load the processor and tokenizer for the model
processor = AutoProcessor.from_pretrained(MODEL_PATH)
tokenizer = processor

# Configure 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the model with the specified quantization configuration
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"✅ Model loaded: {next(model.parameters()).device}")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

✅ Model loaded: cuda:0


In [ ]:
# Example of how to get a response from the model
messages = [
    {"role": "user", "content": "What is machine learning? Explain in 3 lines."}
]

# Step 1: Create the chat template text without tokenizing yet
text = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False  # Important: only get the text at this stage
)

# Step 2: Tokenize the prepared text and move it to the model's device
inputs = processor(
    text=text,
    return_tensors="pt"
).to(model.device)

# Step 3: Generate the model's response
outputs = model.generate(**inputs, max_new_tokens=200)

# Step 4: Decode only the new part of the response (excluding the input)
response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print("🤖 Gemma 4:", response)

🤖 Gemma 4: Machine learning is a field where computers learn patterns and make predictions from data without being explicitly programmed. It involves training algorithms on datasets to identify trends and make informed decisions. Essentially, it allows machines to learn from experience, improving performance over time.


In [ ]:
# Define an interactive chat function
def start_chat():
    chat_history = []
    print("💬 Gemma 4 Chat (type 'quit' to exit)\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            print("Bye!")
            break

        # Add user's message to chat history
        chat_history.append({"role": "user", "content": user_input})

        # Step 1: Apply chat template to history to get raw text
        text = processor.apply_chat_template(
            chat_history,
            add_generation_prompt=True,
            tokenize=False  # Ensure it returns text only
        )

        # Step 2: Tokenize the text and move it to the model's device
        inputs = processor(
            text=text,
            return_tensors="pt"
        ).to(model.device)  # Use model.device for correct placement

        # Step 3: Generate the model's response
        outputs = model.generate(
            **inputs,
            max_new_tokens=200
        )

        # Step 4: Decode only the newly generated part of the response
        response = processor.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        # Add model's response to chat history
        chat_history.append({"role": "assistant", "content": response})
        print(f"🤖 Gemma 4: {response}\n")

# Start the interactive chat
start_chat()

💬 Gemma 4 Chat (type 'quit' to exit)

You: hi
🤖 Gemma 4: Hi! How can I help you today? 😊

You: what is ipl
🤖 Gemma 4: **IPL stands for the Indian Premier League.**

It is the **premier Twenty20 (T20) professional Twenty20 cricket league** in the world, and it is a massive, high-profile franchise-based tournament.

Here's a breakdown of what it is and why it's so popular:

### Key Things to Know About the IPL:

1. **What is it?** It's a professional cricket league where several teams (franchises) compete against each other in a short, fast-paced format (T20 cricket) for a championship title.
2. **Format:** Matches are short, intense, and usually played over a few hours. This makes it very exciting to watch, as there are many opportunities for big scores and dramatic finishes.
3. **Teams:** The league features 10 to 12 teams, each representing a different city or region in India. These teams compete based on their performance throughout the season

You: can you explain about Dhurandhar T